# Cohort Retention Analysis — Logo Retention
## Business Question
What % of customers acquired in each month are still active after 1, 3, 6, 12 months?
Where is the largest single-month drop (the "leaky bucket")?

**Logo retention** tracks *customers*, not revenue.
Compare with Notebook 02 (Revenue Retention / NRR) to understand the full picture.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", font_scale=1.1)

ROOT  = Path("..")
IMG   = ROOT / "images"
cust  = pd.read_csv(ROOT / "data" / "customers.csv")
events= pd.read_csv(ROOT / "data" / "subscription_events.csv")

COLORS = {"starter":"#e9c46a","pro":"#2a9d8f","enterprise":"#264653"}
MRR_MAP = {"starter":29,"pro":79,"enterprise":199}
print(f"Customers: {len(cust):,} | Events: {len(events):,}")

## 1 · Build Customer Cohort Table

In [ ]:
# Active customers per cohort per calendar month
active = events[events["event"].isin(["active","expansion","new"])].copy()
active = active.merge(cust[["customer_id","acquisition_month"]], on="customer_id")

# Cohort index = months since acquisition
def cohort_index(row):
    acq = pd.Period(row["acquisition_month"], "M")
    cur = pd.Period(row["month"], "M")
    return (cur - acq).n

active["cohort_index"] = active.apply(cohort_index, axis=1)

cohort_table = (
    active.groupby(["acquisition_month","cohort_index"])["customer_id"]
    .nunique()
    .reset_index()
)
cohort_pivot = cohort_table.pivot(
    index="acquisition_month", columns="cohort_index", values="customer_id"
)
cohort_sizes  = cohort_pivot[0]
retention_pct = cohort_pivot.divide(cohort_sizes, axis=0).mul(100)

print(f"Cohorts: {len(retention_pct)}  |  Max follow-up months: {retention_pct.columns.max()}")
retention_pct.iloc[:,:7].round(1)

## 2 · Retention Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
data = retention_pct.iloc[:, :13].round(1)

sns.heatmap(
    data, annot=True, fmt=".0f",
    cmap="RdYlGn", vmin=0, vmax=100,
    linewidths=0.4, linecolor="#e5e5e5",
    cbar_kws={"label": "Logo Retention %", "shrink": 0.6},
    ax=ax,
)
ax.set_title("Customer Cohort Retention Heatmap — SaaS Product (2022–2023)",
             fontsize=14, pad=16)
ax.set_xlabel("Months Since Acquisition (Cohort Index)")
ax.set_ylabel("Acquisition Cohort")
ax.yaxis.set_tick_params(rotation=0)
plt.tight_layout()
plt.savefig(IMG / "01_logo_retention_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → images/01_logo_retention_heatmap.png")

## 3 · Average Retention Curve

In [ ]:
avg = retention_pct.mean().dropna()

fig, ax = plt.subplots(figsize=(12, 5))
for cohort in retention_pct.index:
    row = retention_pct.loc[cohort].dropna()
    ax.plot(row.index, row.values, alpha=0.2, color="#264653", linewidth=1)

ax.plot(avg.index, avg.values, color="#2a9d8f", linewidth=3,
        label="Average retention", zorder=5)
ax.set_xlabel("Months Since Acquisition")
ax.set_ylabel("Logo Retention (%)")
ax.set_title("Retention Curves — Individual Cohorts + Average")
ax.legend()
plt.tight_layout()
plt.savefig(IMG / "02_logo_retention_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print("Average retention by month:")
for mo, v in avg.items():
    bar = "█" * max(0, int(v / 3))
    print(f"  M{mo:>2}: {v:5.1f}%  {bar}")

## 4 · Cohort Quality — M3 Retention by Cohort

In [ ]:
if 3 in retention_pct.columns:
    m3 = retention_pct[3].dropna().sort_values()
    worst = m3.head(5)
    best  = m3.tail(5)

    fig, ax = plt.subplots(figsize=(11, 4))
    colors = ["#e76f51" if v < m3.mean() else "#2a9d8f" for v in m3.values]
    ax.bar(m3.index.astype(str), m3.values, color=colors, edgecolor="white")
    ax.axhline(m3.mean(), color="#264653", linestyle="--",
               label=f"Average {m3.mean():.1f}%")
    ax.set_title("M3 Logo Retention by Acquisition Cohort")
    ax.set_ylabel("Retention at Month 3 (%)")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    plt.tight_layout()
    plt.savefig(IMG / "03_m3_cohort_quality.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"M3 retention: avg={m3.mean():.1f}%  best={m3.max():.1f}%  worst={m3.min():.1f}%")

## Key Findings
- Largest drop is **Month 0 → Month 1** (typical SaaS pattern — new users churn fast)
- Retention stabilises after Month 3–4 (customers who stay past 3 months are long-term)
- Q4 cohorts (Oct–Nov) tend to have better M3 retention — likely higher-intent buyers
- **Action:** Focus onboarding improvements on the first 30–90 days → highest leverage point